# City Zone Creation

## Zone Network Functions

### Setting Up

In [1]:
%load_ext autoreload
%reload_ext autoreload

import pandas as pd
import numpy as np
import geopandas as gpd
import networkx as nx
import shapely
import folium
import osmnx as ox
from rtree import index as rtree_index
import os
import random

from utils.route_utils import degrees_to_meters, meters_to_degrees
from utils.checkpoint_utils import save_data

from shapely.geometry import LineString, Point
from shapely import wkt

from __future__ import absolute_import, division

ox.settings.log_console=True
ox.settings.use_cache=True

In [2]:
# For reproducibility. Makes any random sampling deterministic.
SEED = 42 # You can change this value to any integer depending on the run
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED);
np.random.seed(SEED)

#### Graph and Features

In [3]:
def prepare_city_features(city_name):
    location = f"{city_name}, Philippines"
    graph_dir = f"../data/street_networks/{city_name}.graphml"

    if os.path.exists(graph_dir):
        graph = ox.load_graphml(graph_dir)
    else:
        graph = ox.graph_from_place(location, network_type="drive")
        ox.save_graphml(graph, graph_dir)

    print(f"Preparing city features for {location}...")
    print("No. of edges: ", graph.number_of_edges())
    print("No. of nodes: ", graph.number_of_nodes())

    # Filter Roads
    road = ox.graph_to_gdfs(graph, nodes=False, edges=True)
    filtered_roads = road[road['junction'].isna()]

    rows_with_lists = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, list))]
    rows_with_strings = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, str))]

    filter_options = ['primary', 'secondary', 'tertiary', 'trunk', 'unclassified']

    roads_str = rows_with_strings.loc[rows_with_strings['highway'].isin(filter_options)]
    roads_lst = rows_with_lists[rows_with_lists['highway'].apply(lambda lst: any(x in filter_options for x in lst))]

    # Filter Waterways
    waterways = ox.features_from_place(location, tags={'waterway': True})
    rivers = waterways[waterways['waterway'].isin(['river'])]
    streams = waterways[waterways['waterway'].isin(['stream'])]

    return {
        "roads_str": roads_str, "roads_str_idx": roads_str.sindex,
        "roads_lst": roads_lst, "roads_lst_idx": roads_lst.sindex,
        "rivers": rivers, "rivers_idx": rivers.sindex,
        "streams": streams, "streams_idx": streams.sindex
    }

In [ ]:
# LEVEL 1 - COMBINE AMENITIES BY POLYGON
# Creates a copy of a graph and connects non-contiguous and non-overlapping shapes instead of merging
def combine_amenities_by_polygon(graph, max_distance, max_perimeter, city_features):
    combined_graph = nx.Graph()
    list_to_merge = []
    idx_poly = rtree_index.Index()
    idx_all = rtree_index.Index()
    poly_cache = {}
    id_to_key = {}
    insert_id = 0

    # Pre-compute and cache node geometries and properties
    for node_key, node_data in graph.nodes.items():
        idx_all.insert(insert_id, node_data['geometry'].bounds)
        id_to_key[insert_id] = node_key
        insert_id += 1

        if 'geometry' in node_data and node_data['geometry'].geom_type in ['Polygon', 'MultiPolygon']:
            enlarged_polygon = node_data['geometry'].buffer(meters_to_degrees(max_distance))
            numeric_key = int(node_key.split('_')[1])
            bounds = enlarged_polygon.bounds
            bounds_float = tuple(float(coord) for coord in bounds)
            idx_poly.insert(numeric_key, bounds_float)

            poly_cache[node_key] = {
                'geometry': node_data['geometry'],
                'centroid': node_data['geometry'].centroid,
                'amenity': node_data['amenity'],
                'name': node_data.get('name', ''),
                'points': node_data.get('amenity_points', 0),
                'lat': node_data['lat'],
                'lon': node_data['lon']
            }

    for node_key, node_data in poly_cache.items():
        nodes_to_merge = []

        #Check first if it is already in a list of polygons to be merged
        for merge_list in list_to_merge:
            if node_key in merge_list:
                nodes_to_merge = merge_list
                break

        # If this is a new node that is not part of any list, add itself to the list for merging later
        if not nodes_to_merge:
            nodes_to_merge.append(node_key)

        total_distance = 0
        combined_node = node_data['geometry']

        # Find potential neighbors using R-tree
        for other_node_key in idx_poly.intersection(node_data['geometry'].bounds):
            formatted_key = f"polygon_{other_node_key}"
            other_node_data = poly_cache[formatted_key]
            if (formatted_key == node_key or node_data['amenity'] != other_node_data['amenity']):
                continue

            # Quick distance check
            distance = degrees_to_meters(node_data['geometry'].distance(other_node_data['geometry']))
            if distance > max_distance:
                continue

            # Check for intersecting features
            amenities_intersecting = False
            line_between_centroids = LineString([node_data['centroid'], other_node_data['centroid']])
            for hit in idx_all.intersection(line_between_centroids.bounds):
                hit_key = id_to_key.get(hit)
                if hit_key in (node_key, formatted_key) or graph.nodes[hit_key]['amenity'] == node_data['amenity']:
                    continue
                if graph.nodes[hit_key]['geometry'].intersects(line_between_centroids):
                    amenities_intersecting = True
                    break

            # Check combined perimeter
            combined_node = shapely.ops.unary_union([combined_node, other_node_data['geometry']])
            total_distance += degrees_to_meters(combined_node.length)

            if not amenities_intersecting and total_distance < max_perimeter and not find_intersecting_features(line_between_centroids, city_features):
                nodes_to_merge.append(formatted_key)

        # Add group to merge list if more than one node
        if nodes_to_merge not in list_to_merge:
            list_to_merge.append(nodes_to_merge) # Add to the list to merge the polygons later

    temp_graph = to_graph(list_to_merge)
    lists = graph_to_list(temp_graph)

    # Now we will finally connect all polygons in the list
    for merge_list in lists:
        first = True
        for node_key in merge_list:
            if first:
                combined_node = graph.nodes[node_key]['geometry']
                combined_node_amenity = graph.nodes[node_key]['amenity']
                combined_node_key = node_key
                combined_node_geometry = graph.nodes[node_key]['geometry']
                combined_node_name = graph.nodes[node_key]['name']
                combined_node_lat = graph.nodes[node_key]['lat']
                combined_node_lon = graph.nodes[node_key]['lon']
                combined_node_points = graph.nodes[node_key]['amenity_points']
                first = False
            else:
                combined_node = shapely.ops.unary_union([combined_node_geometry, graph.nodes[node_key]['geometry']])
                combined_node_geometry = combined_node
                combined_node_name = combine_names(combined_node_name, graph.nodes[node_key].get('name'))
                combined_node_lat = combined_node_geometry.centroid.y
                combined_node_lon = combined_node_geometry.centroid.x
                combined_node_points += graph.nodes[node_key].get('amenity_points', 0)

        combined_graph.add_node(combined_node_key, geometry=combined_node_geometry, name=combined_node_name, lat=combined_node_lat, amenity=combined_node_amenity,
                                lon=combined_node_lon, amenity_points=combined_node_points)

    return combined_graph

In [5]:
# To merge duplicates in lists
def to_graph(nodes):
    G = nx.Graph()
    for part in nodes:
        G.add_nodes_from(part)
        G.add_edges_from(to_edges(part))
    return G

def to_edges(nodes):
    it = iter(nodes)
    last = next(it)

    for current in it:
        yield last, current
        last = current

def graph_to_list(G):
    connected_components = nx.connected_components(G)
    lists = [list(component) for component in connected_components]
    return lists

In [6]:
# This function will find which road or river intersects between amenities
def find_intersecting_features(line, city_features):
    separation_options = ['primary', 'secondary', 'tertiary', 'unclassified']

    # Check intersection with filtered roads
    possible_matches_roads = city_features["roads_str"].iloc[list(city_features["roads_str_idx"].intersection(line.bounds))]
    for index, row in possible_matches_roads.iterrows():
        if line.intersects(row['geometry']) and row['highway'] in separation_options:
            return True

    possible_matches_lists = city_features["roads_lst"].iloc[list(city_features["roads_lst_idx"].intersection(line.bounds))]
    for index, row in possible_matches_lists.iterrows():
        if line.intersects(row['geometry']):
            list_highway = row['highway']
            if any(x in separation_options for x in list_highway):
                return True

    # Check intersection with filtered rivers
    possible_matches_rivers = city_features["rivers"].iloc[list(city_features["rivers_idx"].intersection(line.bounds))]
    for index, row in possible_matches_rivers.iterrows():
        if line.intersects(row['geometry']):
            return True

    # Check intersection with filtered streams
    possible_matches_streams = city_features["streams"].iloc[list(city_features["streams_idx"].intersection(line.bounds))]
    for index, row in possible_matches_streams.iterrows():
        if line.intersects(row['geometry']):
            return True

    return False

### Reading Population Data

In [7]:
def load_pop_data(city_name):
  filepath = f'../data/population_distribution/{city_name.lower()}-population-polygon.csv'
  df = pd.read_csv(filepath)
  df['geometry'] = df['geometry'].apply(wkt.loads)
  return gpd.GeoDataFrame(df, crs='epsg:32651')

### Initial City Network Creation

In [ ]:
# Create a networkx graph
def create_network(amenities_polygon_gdf, amenities_point_gdf):
    """
    Fast network construction:
      - Vectorized centroid/coord extraction
      - add_nodes_from bulk insertion
      - Single upfront validation/filtering
    """

    poly = amenities_polygon_gdf
    pts = amenities_point_gdf
    G = nx.Graph()

    # Validate required columns
    req_poly = {'geometry', 'amenity', 'name', 'addr_city', 'amenity_points'}
    req_point = {'geometry', 'amenity', 'name', 'addr_city'}
    missing_poly = req_poly - set(amenities_polygon_gdf.columns)
    missing_point = req_point - set(amenities_point_gdf.columns)
    if missing_poly:
        raise ValueError(f"Polygon GDF missing columns: {sorted(missing_poly)}")
    if missing_point:
        raise ValueError(f"Point GDF missing columns: {sorted(missing_point)}")

    # Fix invalid geometries
    poly["geometry"] = poly["geometry"].apply(lambda g: g if g.is_valid else g.buffer(0))
    pts["geometry"]  = pts["geometry"].apply(lambda g: g if g.is_valid else g.buffer(0))

    # Filter out invalid geometries
    poly = poly[poly["geometry"].is_valid]
    pts = pts[pts["geometry"].is_valid]

    # Polygons: compute centroids once (vectorized)
    poly_cent = poly.geometry.centroid
    poly_lat = poly_cent.y.to_numpy()
    poly_lon = poly_cent.x.to_numpy()

    # Polygons: compute centroids once (vectorized)
    pts_lat = pts.geometry.y.to_numpy()
    pts_lon = pts.geometry.x.to_numpy()

    # Build (node_id, attrs) tuples for bulk add
    poly_nodes = [
        (f"polygon_{i}", {
            "polygon_index": int(i),
            "geometry": g,
            "lat": float(la),
            "lon": float(lo),
            "amenity": a,
            "name": n or "",
            "addr_city": c,
            "amenity_points": ap
        })
        for i, g, la, lo, a, n, c, ap in zip(
            poly.index,
            poly["geometry"].to_numpy(),
            poly_lat, poly_lon,
            poly["amenity"].to_numpy(),
            poly["name"].to_numpy(),
            poly["addr_city"].to_numpy(),
            poly["amenity_points"].to_numpy()
        )
    ]
    G.add_nodes_from(poly_nodes)

    isTranspo = (pts["amenity"].to_numpy() == "transportation")
    point_nodes = [
        (f"point_{i}", {
            "point_index": int(i),
            "geometry": g,
            "lat": float(la),
            "lon": float(lo),
            "amenity": a,
            "name": n or "",
            "addr_city": c,
            "is_in_polygon": False,
            "isTranspo": bool(t)
        })
        for i, g, la, lo, a, n, c, t in zip(
            pts.index,
            pts["geometry"].to_numpy(),
            pts_lat, pts_lon,
            pts["amenity"].to_numpy(),
            pts["name"].to_numpy(),
            pts["addr_city"].to_numpy(),
            isTranspo
        )
    ]
    G.add_nodes_from(point_nodes)

    return G


### Amenity and Zone Connection

In [ ]:
# Before Level 1 - ONLY FOR Manila
# To filter out the residential areas that do not exceed the threshold

def filter_out_low_population_density_residentials(graph_to_filter, pop_graph):
    filtered_graph = nx.Graph()

    for node_key, node_data in list(graph_to_filter.nodes.items()):
        if 'geometry' in node_data and node_data['geometry'].geom_type in ['Polygon', 'MultiPolygon'] and node_data['amenity'] in ("residential", "residential areas"):
            # Different Keyword
            # is_a_zone - Will be added to the filtered graph as it means that the residential area has pop density above the threshold
            if node_key in pop_graph and pop_graph.nodes[node_key]['is_a_zone']:
                filtered_graph.add_node(node_key, **node_data)

        else: # If a point or any polygon taht is not residential area, add to the graph
            if node_data['amenity'] != "residential areas" and node_data['amenity'] != "residential":
                filtered_graph.add_node(node_key, **node_data)

    # Should return a filtered graph
    return filtered_graph

In [10]:
# LEVEL 2 - Function to connect zones in a network
def create_zone_network(graph, pop_graph, max_distance, city_features):
    connect_graph = nx.Graph()
    network_id = 1
    list_to_connect = []
    idx = rtree_index.Index()

    for node_key, node_data in graph.nodes.items():
        enlarged_polygon = node_data['geometry'].buffer(meters_to_degrees(max_distance))
        bounds = enlarged_polygon.bounds
        bounds_float = tuple(float(coord) for coord in bounds)
        numeric_key = int(node_key.split('_')[1])
        idx.insert(numeric_key, bounds_float)

    for node_key, node_data in list(graph.nodes.items()):
        if 'geometry' in node_data and node_data['geometry'].geom_type in ['Polygon', 'MultiPolygon']:
            connect_nodes = []

            #Check first if it is already in a list of polygons to be connected
            for connect_list in list_to_connect:
                if node_key in connect_list:
                    connect_nodes = connect_list
                    break

            # If this is a new node that is not part of any list, add itself to the list for merging later
            if not connect_nodes:
                connect_nodes.append(node_key)

            # If this is not a residential area that is its own zone
            if node_key not in pop_graph or not pop_graph.nodes[node_key]['is_a_zone']:
                for other_node_key in idx.intersection(node_data['geometry'].bounds):
                    formatted_key = f"polygon_{other_node_key}"
                    other_node_data = graph.nodes[formatted_key]
                    if 'geometry' in other_node_data and other_node_data['geometry'].geom_type in ['Polygon', 'MultiPolygon']:
                        if node_key != formatted_key:

                            distance = degrees_to_meters(node_data['geometry'].distance(other_node_data['geometry']))

                            # Check if they are within distance of each other
                            if distance <= max_distance:
                                line_between_centroids = LineString([node_data['geometry'].centroid, other_node_data['geometry'].centroid])
                                if not find_intersecting_features(line_between_centroids, city_features):
                                    if formatted_key not in pop_graph or not pop_graph.nodes[formatted_key]['is_a_zone']:
                                        connect_nodes.append(formatted_key)

            if connect_nodes not in list_to_connect:
                list_to_connect.append(connect_nodes) # Add to the list to merge the polygons later

    temp_graph = to_graph(list_to_connect)
    lists = graph_to_list(temp_graph)

    # Now we will finally connect all polygons in the list
    for merge_list in lists:
        first = True
        for node_key in merge_list:
            if first:
                combined_node = graph.nodes[node_key]['geometry']
                combined_node_amenity = graph.nodes[node_key]['amenity']
                combined_node_key = node_key
                combined_node_geometry = graph.nodes[node_key]['geometry']
                combined_node_name = graph.nodes[node_key]['name']
                combined_node_lat = graph.nodes[node_key]['lat']
                combined_node_lon = graph.nodes[node_key]['lon']
                combined_node_points = graph.nodes[node_key]['amenity_points']
                first = False
            else:
                combined_node = shapely.ops.unary_union([combined_node_geometry, graph.nodes[node_key]['geometry']])
                combined_node_geometry = combined_node
                combined_node_name = combine_names(combined_node_name, graph.nodes[node_key].get('name'))
                combined_node_lat = combined_node_geometry.centroid.y
                combined_node_lon = combined_node_geometry.centroid.x
                combined_node_points += graph.nodes[node_key].get('amenity_points', 0)

        network_id += 1
        connect_graph.add_node(combined_node_key, geometry=combined_node_geometry, name=combined_node_name, lat=combined_node_lat, amenity=combined_node_amenity,
                                lon=combined_node_lon, amenity_points=combined_node_points, network_id=network_id)

    return connect_graph

def combine_names(name1, name2):
    # Combine names ensuring that no null values are included
    if isinstance(name1, str) and isinstance(name2, str):
        return f"{name1}, {name2}"
    elif isinstance(name1, str):
        return name1
    elif isinstance(name2, str):
        return name2
    else:
        return None

In [ ]:

# Function to check population density - To be used for the zone connection
# Uses the combined graph
# Formula: Population Density = Total Population / Total Area

def check_residential_population_density(graph, threshold, population_gdf):
    # Create an R-tree index for efficient spatial querying
    idx = rtree_index.Index()

    # Populate the R-tree index with points
    for index, row in population_gdf.iterrows():
        idx.insert(index, (row['longitude'], row['latitude'], row['longitude'], row['latitude']))

    pop_graph = nx.Graph()

    for node_key, node_data in graph.nodes.items():
        # Check if its a polygon and is a residential area
        if 'geometry' in node_data and node_data['geometry'].geom_type in ['Polygon', 'MultiPolygon'] and node_data['amenity'] in ("residential", "residential areas"):
            total_pop = 0

            # Query the R-tree index to find points within the polygon
            for point_idx in idx.intersection(node_data['geometry'].bounds):
                point = population_gdf.loc[point_idx]
                if Point(point['longitude'], point['latitude']).within(node_data['geometry']):
                    total_pop += point['phl_general_2020']  # Add the density

            if total_pop > threshold:
                node_data["is_a_zone"] = True
            else:
                node_data["is_a_zone"] = False

            pop_graph.add_node(node_key, density=total_pop, **node_data)
    return pop_graph

### Visualization

In [ ]:
# 1 - Function to plot/visualize main graph network on the map
def plot_network_on_map(amenities_network, initial_location=[0, 0], zoom_start=10):
    # Create a map centered at the initial location
    map_center = (14.599512, 120.984222) # TEMPORARY WILL ZOOM TO MANILA
    m = folium.Map(location=map_center, zoom_start=zoom_start, tiles='openstreetmap')

    #Colours for Visualization
    amenity_colors = {
        'education': 'green',
        'finance': 'blue',
        'government offices': 'red',
        'government areas': 'red',
        'government' : 'red',
        'grocery': 'orange',
        'health': 'magenta',
        'malls': 'yellow',
        'residential areas': 'brown',
        'residential': 'brown',
        'security': 'gray',
        'transportation': 'lightblue',
        'groceries' : 'indigo',
        'mall' : 'violet',
        'transportation': 'lightblue',
        'groceries' : 'indigo',
        'industrial': 'darkorange',
        'institutional': 'teal',
        'commercial': 'gold',
        'miscellaneous': 'dimgray',
        'others': 'black'
    }

    # Iterate over the nodes in the network
    for node, data in amenities_network.nodes(data=True):
        # Check if the node has a geometry attribute
        if 'geometry' in data:
            # Get the geometry of the node
            geometry = data['geometry']

            # Check the geometry type and plot accordingly
            if geometry.geom_type == 'Point':
                # Plot a marker for points
                #folium.Marker(location=[geometry.y, geometry.x], popup=f"{data['name']}").add_to(m)
                continue
            elif geometry.geom_type in ['Polygon', 'MultiPolygon']:
                # Plot polygons or multipolygons
                color = amenity_colors[data.get('amenity')]
                if geometry.geom_type == 'Polygon':
                    polygons = [geometry]
                else:
                    polygons = geometry.geoms

                for polygon in polygons:
                    coordinates = []
                    for point in polygon.exterior.coords:
                        coordinates.append([point[1], point[0]])
                    folium.Polygon(locations=coordinates, fill=True, color=color, fill_opacity=0.4).add_to(m)

    # Return the map
    return m


In [13]:
# This is to visualize the stops
def plot_stops_on_map(network_map, stops, initial_location=[0, 0], zoom_start=10):
    # Iterate over the nodes in the network
    for stop in stops:
        folium.Marker(location=[stop.lat, stop.long], popup=f"{stop.isTranspo}").add_to(network_map)

    return network_map

In [14]:
# 2 - Function to plot/visualize connected zones on the map

# This is to better visualize the networks
def plot_connected_zones_network_on_map(graph, initial_location=[0, 0], zoom_start=10):
    # Create a map centered at the initial location
    map_center = (14.599512, 120.984222) # TEMPORARY WILL ZOOM TO MANILA
    m = folium.Map(location=map_center, zoom_start=zoom_start, tiles='openstreetmap')

    #Colours for Visualization
    colors = [
    "Red", "Green", "Blue", "Yellow", "Orange", "Purple", "Cyan", "Magenta", "Maroon",
    "Olive", "Lime", "Teal", "Navy", "Aqua", "Fuchsia", "Coral", "Indigo", "Violet"]

    color_map = {}

    # Iterate over the nodes in the network
    for node, data in graph.nodes(data=True):
        # Check if the node has a geometry attribute
        if 'geometry' in data:
            # Get the geometry of the node
            geometry = data['geometry']

            # Check the geometry type and plot accordingly
            if geometry.geom_type == 'Point':
                # Plot a marker for points
                #folium.Marker(location=[geometry.y, geometry.x], popup=f"{data['name']}").add_to(m)
                continue
            elif geometry.geom_type in ['Polygon', 'MultiPolygon']:

                network_id = data["network_id"]

                if network_id not in color_map:
                    color = random.choice(colors)
                    color_map[network_id] = color
                else:
                    color = color_map[network_id]

                if geometry.geom_type == 'Polygon':
                    polygons = [geometry]
                else:
                    polygons = geometry.geoms

                for polygon in polygons:
                    coordinates = []
                    for point in polygon.exterior.coords:
                        coordinates.append([point[1], point[0]])
                    folium.Polygon(locations=coordinates, fill=True, color=color, fill_opacity=0.4).add_to(m)

    # Return the map
    return m


In [15]:
# 2.5 - Function to plot/visualize residential areas based on population density on the map
# This is to better visualize which residential areas can become zones
def plot_population_zones_map(graph, initial_location=[0, 0], zoom_start=10):
    # Create a map centered at the initial location
    map_center = (14.599512, 120.984222) # TEMPORARY WILL ZOOM TO MANILA
    m = folium.Map(location=map_center, zoom_start=zoom_start, tiles='openstreetmap')

    # Iterate over the nodes in the network
    for node, data in graph.nodes(data=True):
        # Check if the node has a geometry attribute
        if 'geometry' in data:
            # Get the geometry of the node
            geometry = data['geometry']

            # Check the geometry type and plot accordingly
            if geometry.geom_type == 'Point':
                # Plot a marker for points
                #folium.Marker(location=[geometry.y, geometry.x], popup=f"{data['name']}").add_to(m)
                continue
            elif geometry.geom_type in ['Polygon', 'MultiPolygon']:
                if geometry.geom_type == 'Polygon':
                    polygons = [geometry]
                else:
                    polygons = geometry.geoms

                for polygon in polygons:
                    coordinates = []
                    for point in polygon.exterior.coords:
                        coordinates.append([point[1], point[0]])

                    if (data['is_a_zone']):
                        folium.Polygon(locations=coordinates, fill=True, color="green", fill_opacity=0.4).add_to(m)
                    else:
                        folium.Polygon(locations=coordinates, fill=True, color="red", fill_opacity=0.4).add_to(m)

    # Return the map
    return m

## Per City Processing

In [ ]:
def process_city(city_name, population_gdf, threshold_pop=120):
    print(f"--- Processing {city_name} ---")

    city_features = prepare_city_features(city_name)

    amenity_dir = f"../data/amenity_maps/Taguig" if (city_name == "Taguig") else f"../data/amenity_maps/Jacob-Group/{city_name} City"
    point_file = os.path.join(amenity_dir, f"{city_name}_point.geojson")
    poly_file = os.path.join(amenity_dir, f"{city_name}_polygon.geojson")

    pikl_filepath = f"../data/saved_networks/{city_name}/"
    map_filepath = f"../plots/Zone Creation/{city_name}/"

    os.makedirs(pikl_filepath, exist_ok=True)
    os.makedirs(map_filepath, exist_ok=True)

    # Load Data
    print(f"Loading amenities for {city_name}...")

    points_gdf = gpd.read_file(point_file)
    polygons_gdf = gpd.read_file(poly_file)
    polygons_gdf['amenity_points'] = None

    # Fast spatial index for points inside polygons
    idx = rtree_index.Index()
    for j, point in points_gdf.iterrows():
        idx.insert(j, point['geometry'].bounds)

    for i, polygon in polygons_gdf.iterrows():
        points_within_polygon = [
            j for j in idx.intersection(polygon['geometry'].bounds)
            if polygon['geometry'].intersects(points_gdf.loc[j, 'geometry'])
        ]
        polygons_gdf.at[i, 'amenity_points'] = points_within_polygon

    # 1. Initial Network
    network = create_network(polygons_gdf, points_gdf)
    plot_network_on_map(network, initial_location=[0, 0], zoom_start=100).save(f'{map_filepath}merge_map.html')
    save_data(network, f'{pikl_filepath}{city_name}_Initial_Network.pkl')

    # 2. Filter Residential (Only applied if required)
    if city_name == "Manila":
        filter_residential_graph = check_residential_population_density(network, threshold_pop, population_gdf)
        network = filter_out_low_population_density_residentials(network, filter_residential_graph)
        plot_network_on_map(network, initial_location=[0, 0], zoom_start=100).save(f'{map_filepath}residential_filter.html')
        save_data(filter_residential_graph, f'{pikl_filepath}{city_name}_Population_Graph.pkl')
        save_data(network, f'{pikl_filepath}{city_name}_Filtered_Network.pkl')

    print(f"Combining amenities by polygon for {city_name} (1/2)...")
    # 3. Combine Amenities (Level 1)
    combined_graph = combine_amenities_by_polygon(network, max_distance=100, max_perimeter=10000, city_features=city_features)
    plot_network_on_map(combined_graph, initial_location=[0, 0], zoom_start=100).save(f'{map_filepath}after_map.html')
    save_data(combined_graph, f'{pikl_filepath}{city_name}_Combined_Amenities_Network.pkl')

    # 4. Population Zones
    pop_graph = check_residential_population_density(combined_graph, 100, population_gdf)
    # pop_map = plot_population_zones_map(pop_graph, initial_location=[0, 0], zoom_start=100) # Optional save
    save_data(pop_graph, f'{pikl_filepath}{city_name}_Population_Graph.pkl')

    print(f"Creating a zone network for {city_name} (2/2)...")
    # 5. Zone Networks (Level 2)
    zone_network = create_zone_network(graph=combined_graph, pop_graph=pop_graph, max_distance=100, city_features=city_features)
    plot_connected_zones_network_on_map(zone_network, initial_location=[0, 0], zoom_start=100).save(f'{map_filepath}networks_map.html')
    save_data(zone_network, f'{pikl_filepath}{city_name}_Zone_Network.pkl')

    print(f"Finished processing {city_name}.")

In [ ]:
manila_population_gdf = load_pop_data("manila")
makati_population_gdf = load_pop_data("makati")
mandaluyong_population_gdf = load_pop_data("mandaluyong")
taguig_population_gdf = load_pop_data("taguig")

In [ ]:
cities_to_process = [
    ("Manila", manila_population_gdf, 120),
    ("Makati", makati_population_gdf, 100),
    ("Mandaluyong", mandaluyong_population_gdf, 100)
]

for city, pop_gdf, threshold in cities_to_process:
    process_city(city, pop_gdf, threshold_pop=threshold)

--- Processing Manila ---
Preparing city features for Manila, Philippines...
No. of edges:  12739
No. of nodes:  5062
Loading amenities for Manila...


/tmp/ipykernel_2251/470888747.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poly_cent = poly.geometry.centroid


Combining amenities by polygon for Manila (1/2)...
Creating a zone network for Manila (2/2)...
Finished processing Manila.
--- Processing Makati ---
Preparing city features for Makati, Philippines...
No. of edges:  5558
No. of nodes:  2294
Loading amenities for Makati...


/tmp/ipykernel_2251/470888747.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poly_cent = poly.geometry.centroid


Combining amenities by polygon for Makati (1/2)...
Creating a zone network for Makati (2/2)...
